# Simulando Dados Sensíveis para Preservar os Dados dos Participantes do PBA
Com o objetivo de publicar os códigos utilizados para as análises do Programa Brasil Alfabetizado (PBA) no GitHub em total conformidade com a Lei Geral de Proteção de Dados (LGPD), bem como com as regras de *compliance* da Fundação Getulio Vargas, o presente código foi estruturado para anonimizar os dados sensíveis dos participantes sem prejudicar o resultado estatístico final das análises.

Sempre que uma nova rodada de análise for iniciada ou novos arquivos forem recebidos, este script deve ser executado previamente para garantir a segurança da base de dados pública.

## 📘 Guia de Uso: Script de Anonimização Persistente e Dinâmica
Este documento explica o funcionamento e a parametrização do ecossistema em Python para o mascaramento de dados pessoais identificáveis (PII).

### 🎯 Objetivo do Script
O script realiza a varredura em lote de arquivos nas extensões `.xlsx` (Excel) e `.csv` dentro do diretório de origem. Ele identifica múltiplos tipos de informações sensíveis e as substitui por dados fictícios realistas gerados deterministicamente pela biblioteca `Faker`.

**Diferenciais desta solução:**

- **Consistência Referencial Global (De-Para):** Se um participante ou funcionário aparece em diferentes arquivos exercendo papéis distintos (ex: como Alfabetizador em uma planilha e Coordenador em outra), o algoritmo garante que ele receberá exatamente o mesmo nome, CPF e dados fakes em todo o ecossistema de arquivos.

- **Persistência de Estado (Stateful):** Os mapeamentos são salvos localmente em arquivos de dicionário (`.csv`). Isso funciona como uma "memória" para o script: se novos arquivos forem adicionados no futuro, as identidades fictícias criadas anteriormente serão rigorosamente mantidas.

- **Preservação de Dados Nulos:** Células vazias originais (como participantes temporariamente sem CPF registrado) permanecem estritamente vazias (`NaN`) na pasta pública, evitando distorções na análise.

### 🛠️ Pré-requisitos
Certifique-se de que o ambiente possui o Python e os pacotes necessários instalados:

---
```bash
pip install pandas faker openpyxl
```
---

> 📌 *Nota: A biblioteca `openpyxl` é indispensável para que o Pandas gerencie o fluxo de leitura e escrita dos arquivos modernos do Excel (`.xlsx`).*

### 📂 Estrutura de Pastas Recomendada
Para que o pipeline funcione de forma automatizada sem a necessidade de reescrever caminhos locais, organize o diretório do seu projeto seguindo o modelo abaixo:

---

```bash
seu_projeto/
│
├── protect_sensitive_data.ipynb  # Este Jupyter Notebook
├── .gitignore                    # Arquivo de proteção do Git
├── dicionario_*.csv              # Mapas de tradução gerados pelo script (Ocultos no Git)
│
└── Data_files/
    ├── original_data/            # ⚠️ COLOQUE SEUS ARQUIVOS REAIS E SENSÍVEIS AQUI
    │   ├── turmas_2025.xlsx
    │   └── cadastros.csv
    │
    └── public_data/              # ✨ OS ARQUIVOS LIMPOS E SEGUROS SERÃO GERADOS AQUI
```

---

### ⚙️ Como Configurar e Customizar (CONFIG)
Toda a inteligência e o mapeamento de colunas do script estão centralizados no dicionário estruturado `CONFIG` na célula de código a seguir. Você não precisa alterar a lógica interna dos loops se novas colunas surgirem; basta atualizar a estrutura de configuração.

Cada bloco do dicionário segue o padrão:

---

```bash
'grupo_de_dados': {
    'colunas': ['Lista', 'de', 'Cabeçalhos', 'da', 'Planilha'],
    'gerador': fake.metodo_faker,
    'arquivo': 'nome_do_dicionario_persistente.csv'
}
```

---

**Regras de Customização:**
1. **Adicionar novos campos:** Se uma nova coluna sensível aparecer (ex: `Email`), basta criar uma nova chave no dicionário apontando para `fake.email` e definir um nome de arquivo `.csv`.

2. **Ignorar colunas para Análises Especiais:** Se for necessário manter dados reais de uma coluna para fins específicos (como manter a coluna `Bairro` intacta para a realização de análises geoespaciais), basta remover por completo o bloco correspondente de dentro do dicionário `CONFIG`. O script ignorará a coluna e ela passará inalterada para a pasta pública.

### 🚀 Como Executar
1. Deposite todos os relatórios brutos e planilhas de monitoramento dentro da pasta `Data_files/original_data/`.

2. Execute todas as células deste notebook em sequência.

**O que acontece nos bastidores após a execução?**

O script varre a pasta de origem, atualiza os dicionários locais na raiz do projeto com quaisquer novos registros identificados e grava cópias idênticas dos arquivos originais na pasta `Data_files/public_data/`, porém com o conteúdo das colunas configuradas totalmente anonimizado.

### 🔒 Segurança Crítica (`.gitignore`)
> **ATENÇÃO ABSOLUTA:** Os arquivos contidos na pasta `original_data` e os dicionários gerados (`dicionario_*.csv`) carregam o mapeamento direto que desfaz o sigilo dos participantes. Eles **NUNCA** podem ser commitados ou enviados para um repositório público no GitHub.

Certifique-se de que o arquivo `.gitignore` na raiz do seu projeto contenha explicitamente as seguintes restrições de rastreamento:

---

```bash
# Ignorar pasta com dados sensíveis originais
Data_files/original_data/

# Ignorar todos os dicionários de tradução e chaves de anonimização
dicionario_nomes.csv
dicionario_cpfs.csv
dicionario_telefones.csv
dicionario_datas.csv
dicionario_bairros.csv
dicionario_enderecos.csv

# Ignorar arquivos temporários do sistema, do Jupyter e travas do Excel
~$*.xlsx
.ipynb_checkpoints/
```

---

### 🔄 Tratamento de Inconsistências e Correções Cadastrais
É normal que ao longo do ciclo de relatórios do programa ocorram correções de digitação nas planilhas originais (ex: o nome real do alfabetizador mudar de "Mariano Sila" para "Mariano Silva"). O script tratará a alteração como uma pessoa nova e gerará uma segunda identidade fictícia.

**Para unificar o histórico manualmente:**

1. Abra o arquivo `dicionario_nomes.csv` gerado na raiz do projeto utilizando o Excel ou um editor de texto.

2. Localize as linhas correspondentes às duas variações do nome real.

3. Altere manualmente a coluna `Falso` da linha corrigida para que ambas apontem para o mesmíssimo nome fictício.

4. Salve o arquivo e reexecute este notebook. O mapeamento manual será respeitado e propagado para todas as análises futuras.

In [3]:
import os
import pandas as pd
from faker import Faker

# 1. Configurações Iniciais
fake = Faker('pt_BR')

# ==============================================================================
# CONFIGURAÇÃO DE ANONIMIZAÇÃO: Centralize aqui todas as colunas que deseja proteger
# ==============================================================================

CONFIG = {
    'nomes': {
        'colunas': ['alfabetizando', 'alfabetizador', 'Alfabetizador', 'ALFABETIZADOR', 'coordenador', 'Coordenador'],
        'gerador': fake.name,
        'arquivo': 'dicionario_nomes.csv'
    },
    'cpfs': {
        'colunas': ['cpf'],
        'gerador': fake.cpf,
        'arquivo': 'dicionario_cpfs.csv'
    },
    'telefones': {
        'colunas': ['Telefone'],
        'gerador': fake.phone_number,
        'arquivo': 'dicionario_telefones.csv'
    },
    'datas_nasc': {
        'colunas': ['Data de nascimento'],
        # Gera uma data fictícia formatada como DD/MM/AAAA para manter o padrão do Excel
        'gerador': lambda: fake.date_of_birth(minimum_age=15, maximum_age=90).strftime('%d/%m/%Y'),
        'arquivo': 'dicionario_datas.csv'
    },
    'bairros': {
        'colunas': ['Bairro'],
        'gerador': fake.bairro,
        'arquivo': 'dicionario_bairros.csv'
    },
    'enderecos': {
        'colunas': ['Endereço completo'],
        'gerador': fake.address,
        'arquivo': 'dicionario_enderecos.csv'
    }
}


In [4]:
# Definindo caminhos para os dados
pasta_origem = "data_files/original_data/"
pasta_destino = "data_files/public_data/"
os.makedirs(pasta_destino, exist_ok=True)

# 2. Carregar todos os dicionários existentes para a memória de forma dinâmica
map_globais = {}
for grupo, info in CONFIG.items():
    if os.path.exists(info['arquivo']):
        df_dict = pd.read_csv(info['arquivo'], dtype=str)
        map_globais[grupo] = dict(zip(df_dict['Real'], df_dict['Falso']))
    else:
        map_globais[grupo] = {}

# 3. Função única e dinâmica para atualizar os mapeamentos
def atualizar_dicionarios_dinamico(df):
    for grupo, info in CONFIG.items():
        for col in info['colunas']:
            if col in df.columns:
                # Extrai valores reais únicos ignorando os nulos
                valores_atuais = df[col].dropna().astype(str).unique()
                for val in valores_atuais:
                    if val not in map_globais[grupo]:
                        # Executa a função geradora correspondente do Faker
                        map_globais[grupo][val] = info['gerador']()

# 4. Função para salvar todos os dicionários atualizados de volta no disco
def salvar_dicionarios_dinamico():
    for grupo, info in CONFIG.items():
        df_salvar = pd.DataFrame(list(map_globais[grupo].items()), columns=['Real', 'Falso'])
        df_salvar.to_csv(info['arquivo'], index=False)

# 5. Loop Principal: Ler, Anonimizar e Salvar
arquivos = [f for f in os.listdir(pasta_origem) if f.lower().endswith(('.xlsx', '.csv'))]

for nome_arquivo in arquivos:
    caminho_completo = os.path.join(pasta_origem, nome_arquivo)
    extensao = os.path.splitext(nome_arquivo)[1].lower()
    
    # A: Leitura dinâmica baseada na extensão
    if extensao == '.xlsx':
        df = pd.read_excel(caminho_completo, dtype=str) 
    elif extensao == '.csv':
        df = pd.read_csv(caminho_completo, dtype=str)
    
    # Passo B: Descobrir dados novos e atualizar os dicionários na memória
    atualizar_dicionarios_dinamico(df)
    
    # Passo C: Aplicar a tradução no DataFrame para cada grupo configurado
    for grupo, info in CONFIG.items():
        for col in info['colunas']:
            if col in df.columns:
                df[col] = df[col].astype(str).map(map_globais[grupo])
            
    # D: Salvamento dinâmico baseado na extensão
    if extensao == '.xlsx':
        df.to_excel(os.path.join(pasta_destino, nome_arquivo), index=False)
    elif extensao == '.csv':
        df.to_csv(os.path.join(pasta_destino, nome_arquivo), index=False, encoding='utf-8')
        
    print(f"✅ {nome_arquivo} processado e salvo na pasta pública.")

# Salva as atualizações de todos os arquivos de mapeamento de uma vez só
salvar_dicionarios_dinamico()
print("🔒 Todos os dicionários persistentes foram atualizados e salvos com sucesso!")


✅ dados_pedagogicos_11052026.csv processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_11052026.xlsx processado e salvo na pasta pública.
✅ turmas_11052026.xlsx processado e salvo na pasta pública.
✅ dados_pedagogicos_25052026.csv processado e salvo na pasta pública.
✅ relatorios_turmas_formativa_1_25052026.xlsx processado e salvo na pasta pública.
✅ turmas_25052026.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_formativa_1_26052026.xlsx processado e salvo na pasta pública.
✅ turmas_26052026.xlsx processado e salvo na pasta pública.
✅ dados_pedagogicos_26052026_old.csv processado e salvo na pasta pública.
✅ dados_pedagogicos_26052026.csv processado e salvo na pasta pública.
✅ dados_pedagogicos_29052029.csv processado e salvo na pasta pública.
🔒 Todos os dicionários persistentes foram atualizados e salvos com sucesso!
